In [ ]:
# GPT-2 scaling study on Kaggle GPU: GPT-2 (124M) vs GPT-2-Medium (355M),
# both on WikiText-103 -- does the QAT+VQ compression trade hold as scale grows?
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!git clone -q https://github.com/abdurrahmanrussel/QAT-VQ-Compression.git repo
%cd repo
!git checkout -q gpt2medium-wikitext103-qatvq
!git log --oneline -3

In [ ]:
!pip install -q "transformers>=4.40" "datasets>=2.18" scikit-learn matplotlib

In [ ]:
%cd /kaggle/working/repo/gpt2_scaling
import os
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

## GPT-2 (124M) on WikiText-103

In [ ]:
!python train_baseline.py --model gpt2 --epochs 1 --bs 8

In [ ]:
!python run_experiments.py --model gpt2 --sub_dim 2 --K 256 --qat_epochs 1 --ft_lr 1e-5 --seeds 0 1 2 --bs_train 8 --bs_eval 8

In [ ]:
# Check the seed search output above for the actual best seed on THIS run before trusting the default below.
!python finetune_vq.py --model gpt2 --seed 1 --epochs 2 --lr 5e-6 --bs_train 4 --bs_eval 4

## GPT-2-Medium (355M) on WikiText-103
Smaller batch size than the 124M run (16GB VRAM budget for a 3x bigger model).

In [ ]:
!python train_baseline.py --model gpt2-medium --epochs 1 --bs 4 --grad_accum 2

In [ ]:
!python run_experiments.py --model gpt2-medium --sub_dim 2 --K 256 --qat_epochs 1 --ft_lr 1e-5 --seeds 0 1 2 --bs_train 4 --bs_eval 4

In [ ]:
# Check the seed search output above for the actual best seed on THIS run before trusting the default below.
!python finetune_vq.py --model gpt2-medium --seed 1 --epochs 2 --lr 5e-6 --bs_train 2 --bs_eval 2

In [ ]:
!python make_figures.py
!cat artifacts/scaling_table.md

In [ ]:
# Push results back to GitHub. Requires a Kaggle Secret named GITHUB_TOKEN
# (fine-grained PAT, repo=QAT-VQ-Compression, Contents: Read and write).
# If this cell fails (Kaggle's internal secrets service is occasionally flaky),
# results are still recoverable afterward via `kaggle kernels output`.
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")

import subprocess
def sh(cmd):
    print('$', cmd.replace(token, '***') if token in cmd else cmd)
    subprocess.run(cmd, shell=True, check=True)

sh('git config user.email "abdurrahmanrussel77@gmail.com"')
sh('git config user.name "Md Abdur Rahman"')
sh('git add artifacts/gpt2/results.json artifacts/gpt2/results_table.md artifacts/gpt2/figures/ '
   'artifacts/gpt2-medium/results.json artifacts/gpt2-medium/results_table.md artifacts/gpt2-medium/figures/ '
   'artifacts/scaling_table.md artifacts/figures/scaling_comparison.png')
sh('git commit -m "GPT-2 scaling study results (124M vs 355M, WikiText-103) from Kaggle" || echo "nothing to commit"')
sh(f'git push https://{token}@github.com/abdurrahmanrussel/QAT-VQ-Compression.git gpt2medium-wikitext103-qatvq')